In [3]:
import hashlib
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ec

BASE58_ALPHABET = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"

def b58encode(data: bytes) -> str:
    leading_zeros = len(data) - len(data.lstrip(b"\x00"))
    number = int.from_bytes(data, "big")
    encoded = []
    while number:
        number, remainder = divmod(number, 58)
        encoded.append(BASE58_ALPHABET[remainder])
    return "1" * leading_zeros + "".join(reversed(encoded))

def b58decode(address: str) -> bytes:
    number = 0
    for char in address:
        if char not in BASE58_ALPHABET:
            raise ValueError(f"Invalid Base58 character: {char!r}")
        number = number * 58 + BASE58_ALPHABET.index(char)
    number_bytes = number.to_bytes((number.bit_length() + 7) // 8, "big") if number else b""
    leading_ones = len(address) - len(address.lstrip("1"))
    return b"\x00" * leading_ones + number_bytes

private_key = ec.generate_private_key(ec.SECP256K1())
pubkey = private_key.public_key().public_bytes(
    serialization.Encoding.X962,
    serialization.PublicFormat.UncompressedPoint,
)

sha256_hash = hashlib.sha256(pubkey).digest()
hash160 = hashlib.new("ripemd160", sha256_hash).digest()
payload = b"\x00" + hash160
checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
address = b58encode(payload + checksum)

decoded = b58decode(address)
decoded_payload, given_checksum = decoded[:-4], decoded[-4:]
recalculated_checksum = hashlib.sha256(
    hashlib.sha256(decoded_payload).digest()
).digest()[:4]

if given_checksum == recalculated_checksum:
    print("Address is valid:", address)
else:
    print("Address is invalid")

tampered_bytes = decoded[:-1] + bytes([decoded[-1] ^ 1])
tampered_address = b58encode(tampered_bytes)
tampered_decoded = b58decode(tampered_address)
tampered_payload, tampered_checksum = tampered_decoded[:-4], tampered_decoded[-4:]
expected_tampered_checksum = hashlib.sha256(
    hashlib.sha256(tampered_payload).digest()
).digest()[:4]

if tampered_checksum == expected_tampered_checksum:
    print("Tampered address is valid")
else:
    print("Tampered address is invalid")

Address is valid: 18DmRhq5F5NYtnU5tsXrX8jijVLd5wwP8c
Tampered address is invalid
